## Step 1 - load data


In [16]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from pathlib import Path
#Fetch Adult Income from UCI 
#(id=2 is the Adult dataset)
adult = fetch_ucirepo(id=2)

# Features and target come as separate DataFrames
X = adult.data.features
y = adult.data.targets

# Combine them into a single DataFrame
df = pd.concat([X, y], axis=1)

# Save locally 
raw_data_path = Path("../data/processed/adult_income_raw.csv")
raw_data_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(raw_data_path, index=False)
print(f"Saved to {raw_data_path.resolve()}")

Loaded 48842 rows with 15 columns
Target column name: income


## Step 2 

In [18]:
print("Shape:", df.shape)
print("\nTarget distribution:")
print(df["income"]# UCI uses '?' to mark missing values — convert these to NaN so pandas recognizes them
df = df.replace("?", np.nan)

# See how many missing values we have per column
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0]).value_counts())
print("\nSex distribution:")
print(df["sex"].value_counts())
print("\nRace distribution:")
print(df["race"].value_counts())

Shape: (48842, 15)

Target distribution:
income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64

Sex distribution:
sex
Male      32650
Female    16192
Name: count, dtype: int64

Race distribution:
race
White                 41762
Black                  4685
Asian-Pac-Islander     1519
Amer-Indian-Eskimo      470
Other                   406
Name: count, dtype: int64


In [19]:
# Data cleaning
# <=50K, <=50K., >50K, >50K.  (the dots come from the test-file format)
df["income"] = df["income"].str.rstrip(".")

print("Target distribution after cleaning:")
print(df["income"].value_counts())

Target distribution after cleaning:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64


In [20]:
# Convert income to binary: >50K = 1, <=50K = 0
# This is the convention: the "positive" class is the one we care about predicting
df["income"] = (df["income"] == ">50K").astype(int)

print("Target distribution as 0/1:")
print(df["income"].value_counts())
print(f"\nBase rate (proportion earning >50K): {df['income'].mean():.3f}")

Target distribution as 0/1:
income
0    37155
1    11687
Name: count, dtype: int64

Base rate (proportion earning >50K): 0.239


In [21]:
# handling missing values- UCI uses ? for missing values 
df = df.replace("?", np.nan)

# See how many missing values we have per column
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

Missing values per column:
workclass         2799
occupation        2809
native-country     857
dtype: int64


In [22]:
# Decision is drop- 
rows_before = len(df)
df = df.dropna().reset_index(drop=True)
rows_after = len(df)

print(f"Dropped {rows_before - rows_after} rows with missing values")
print(f"Remaining rows: {rows_after}")

Dropped 3620 rows with missing values
Remaining rows: 45222


In [25]:
list(df.columns)

['age',
 'workclass',
 'fnlwgt',
 'education',
 'education-num',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'capital-gain',
 'capital-loss',
 'hours-per-week',
 'native-country',
 'income']

In [26]:
# rename columns
df.columns = df.columns.str.replace("-", "_")

print("Columns:", df.columns.tolist())

Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']


In [27]:
print(df.groupby('sex')['income'].mean().round(3))

sex
Female    0.114
Male      0.312
Name: income, dtype: float64


#### This is important — read it carefully:
 The base rate by sex shows that 11.4% of women earn >$50K vs. 31.3% of men in this dataset. This is a huge gap in the underlying data, not something the model created.
Why this matters for fairness later: when we train a model and look at fairness metrics, some gap is already baked into the dataset itself — it reflects 1994 U.S. labor market conditions, not model error. Fairness metrics like Demographic Parity Ratio will always look bad here because of this base-rate difference. That's exactly why we chose Equal Opportunity Difference (EOD) as our primary metric: EOD only compares qualified people, so it's not punished by uneven base rates.

In [33]:
# save cleaned code
cleaned_path = Path("../data/processed/adult_income_cleaned.csv")
df.to_csv(cleaned_path, index=False)
print(f"Saved cleaned data to {cleaned_path.resolve()}")
print(f"Rows: {len(df)}, Columns: {df.shape[1]}")

Saved cleaned data to /Users/gleicechaves/Documents/GRIDS/ML-Reliability-Efficiency-Toolkit/modules/fairness/data/processed/adult_income_cleaned.csv
Rows: 45222, Columns: 15


## Step 3 - Split data and prepare features

1. Target apart from features. The model predicts income, so income cannot be an input.
2. Sensitive attributes apart from features. We remove sex and race from the model's inputs and keep them in a separate DataFrame. This is the most important fairness decision in this step. If the model uses sex directly, any unfair prediction is obvious — the model simply copied the bias. What we want to check is harder: does the model still treat groups unequally, even without knowing their group? If it does, that means other features (like occupation or hours worked) are acting as proxies for sex and carrying bias in through the back door. This is the realistic fairness question for real-world ML systems.
3. Train/test split, stratified, fixed seed. 80% of the data trains the model. 20% is held back to evaluate it. 
4. Encode categorical features. 

In [36]:

TARGET_COL = "income"                    # what the model predicts
SENSITIVE_COLS = ["sex", "race"]         # used for fairness evaluation only

# Features = everything EXCEPT the target and the sensitive columns
feature_cols = [c for c in df.columns 
                if c not in [TARGET_COL] + SENSITIVE_COLS]

# Build the three separate DataFrames
X = df[feature_cols].copy()              # what the model sees
y = df[TARGET_COL].copy()                # what the model predicts
sensitive = df[SENSITIVE_COLS].copy()    # what we use for fairness checks

# Quick sanity check
print(f"Target column: {TARGET_COL}")
print(f"Sensitive columns: {SENSITIVE_COLS}")
print(f"Feature columns: {feature_cols}")


Target column: income
Sensitive columns: ['sex', 'race']
Feature columns: ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country']


In [37]:
# train /test split 
from sklearn.model_selection import train_test_split

# Split X, y, and sensitive together so all three stay row-aligned
X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive,
    test_size=0.2,        # 20% for testing
    stratify=y,           # preserve positive/negative balance in both splits
    random_state=42       # reproducible split
)

# Confirm the split worked as expected
print(f"Training set: {len(X_train):,} rows")
print(f"Test set:     {len(X_test):,} rows")
print(f"\nTrain positive rate: {y_train.mean():.3f}")
print(f"Test positive rate:  {y_test.mean():.3f}")
print("(Both should match the overall base rate of 0.248 — that's stratification working.)")

Training set: 36,177 rows
Test set:     9,045 rows

Train positive rate: 0.248
Test positive rate:  0.248
(Both should match the overall base rate of 0.248 — that's stratification working.)


In [40]:
from sklearn.preprocessing import OneHotEncoder

# ─── Identify which columns are categorical vs. numeric ──────────────
categorical_cols = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")

# ─── Fit the encoder on the training set ONLY ────────────────────────
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(X_train[categorical_cols])

encoded_col_names = encoder.get_feature_names_out(categorical_cols)

# ─── Apply the same encoding to train AND test ───────────────────────
X_train_final = pd.concat([
    X_train[numeric_cols].reset_index(drop=True),
    pd.DataFrame(encoder.transform(X_train[categorical_cols]),
                 columns=encoded_col_names)
], axis=1)

X_test_final = pd.concat([
    X_test[numeric_cols].reset_index(drop=True),
    pd.DataFrame(encoder.transform(X_test[categorical_cols]),
                 columns=encoded_col_names)
], axis=1)

print(f"\nTraining set shape after encoding: {X_train_final.shape}")
print(f"Test set shape after encoding:     {X_test_final.shape}")
print(f"(Both should have the same number of columns.)")

Categorical columns (6): ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'native_country']
Numeric columns (6): ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

Training set shape after encoding: (36177, 97)
Test set shape after encoding:     (9045, 97)
(Both should have the same number of columns.)


## Train base model 

## Step 4 — Train a baseline logistic regression

Now we train the model that the fairness module will audit. We use **logistic regression** because it's the simplest sensible choice for binary classification, it trains fast, and scikit-learn's implementation is well-documented and reproducible.

A few things to remember from earlier:
- The model only sees `X_train_final` — the encoded features, with `sex` and `race` removed.
- `y_train` is 0/1, where 1 means "earns more than $50K/year."
- Any prediction disparity across groups will come from the model picking up bias through **proxy features** (occupation, hours worked, etc.), not from seeing the group label directly.

We also **standardize** the numeric features first. Logistic regression is sensitive to feature scale — `age` ranges from 17–90 while `fnlwgt` ranges into the hundreds of thousands. Without scaling, the larger-magnitude features would dominate the model's weights. Standardizing (subtracting the mean, dividing by the standard deviation) puts all features on roughly the same scale.

Same rule as before: **fit the scaler on train only**, then apply to both. Prevents leakage.

In [41]:
# Train a baseline logistic regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ─── Standardize features ────────────────────────────────────────────
# Fit the scaler on the TRAINING set only (prevents leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

# ─── Train the logistic regression ───────────────────────────────────
# max_iter=1000 → give the solver enough iterations to converge on this dataset
# random_state=42 → reproducible training
# class_weight=None → do NOT rebalance classes; we want to observe the model's
#                      natural behavior on imbalanced data, bias and all
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight=None
)
model.fit(X_train_scaled, y_train)

print("✓ Model trained successfully")
print(f"Number of features: {X_train_scaled.shape[1]}")
print(f"Training accuracy:  {model.score(X_train_scaled, y_train):.3f}")
print(f"Test accuracy:      {model.score(X_test_scaled, y_test):.3f}")

✓ Model trained successfully
Number of features: 97
Training accuracy:  0.850
Test accuracy:      0.844


In [42]:
# gerenate predictions 
# ─── Generate predictions on the test set ────────────────────────────
y_pred = model.predict(X_test_scaled)                   # 0 or 1 per row
y_proba = model.predict_proba(X_test_scaled)[:, 1]       # probability of class 1

# ─── Build one tidy DataFrame with everything for fairness evaluation ─
# This is what we'll hand to the FairnessPipeline.
# Columns: the actual label, the prediction, the probability, sex, race.
results = pd.DataFrame({
    "income_actual":    y_test.reset_index(drop=True),
    "income_predicted": y_pred,
    "income_probability": y_proba,
    "sex":              sens_test["sex"].reset_index(drop=True),
    "race":             sens_test["race"].reset_index(drop=True),
})

# Show the first few rows to make sure everything lined up
print(f"Total test rows: {len(results):,}")
print(f"\nFirst 5 predictions:")
print(results.head())

Total test rows: 9,045

First 5 predictions:
   income_actual  income_predicted  income_probability     sex   race
0              0                 0            0.002708  Female  White
1              1                 1            0.998333    Male  White
2              0                 0            0.128073  Female  Black
3              1                 0            0.409361    Male  White
4              0                 0            0.068994  Female  Black


In [43]:
# accuracy peek by group
per_group_accuracy = results.groupby("sex").apply(
    lambda g: (g["income_actual"] == g["income_predicted"]).mean(),
    include_groups=False
)

print("Accuracy by sex:")
print(per_group_accuracy.round(3))
print(f"\nAccuracy gap (max - min): {per_group_accuracy.max() - per_group_accuracy.min():.3f}")

Accuracy by sex:
sex
Female    0.916
Male      0.809
dtype: float64

Accuracy gap (max - min): 0.107


 why it's more complex than it looks:
 
The model is more accurate on women than on men. At first glance this sounds like "the model treats women better" — but it isn't that simple, and this is a great teaching moment about why fairness needs more than accuracy.

Remember the base rates: only 11% of women earn >50K, vs 31% of men. For women, simply predicting "no" almost always gives you 89% accuracy. The model gets 92% on women mainly because the easy "no" cases are so common. On men, where the positive class is much more frequent, the model has a harder classification job and lands at 81%.
Same accuracy metric, two very different situations. One group is easy to be right about because one class dominates; the other is harder. This is exactly why EOD (Equal Opportunity Difference) is the right fairness metric here — it looks only at the qualified people and asks whether the model identifies them equally well across groups. 

In [45]:
# Save the 
predictions_path = Path("../data/processed/test_predictions.csv")
results.to_csv(predictions_path, index=False)

print(f"✓ Predictions saved to {predictions_path.resolve()}")
print(f"  Rows: {len(results):,}")
print(f"  Columns: {list(results.columns)}")

✓ Predictions saved to /Users/gleicechaves/Documents/GRIDS/ML-Reliability-Efficiency-Toolkit/modules/fairness/data/processed/test_predictions.csv
  Rows: 9,045
  Columns: ['income_actual', 'income_predicted', 'income_probability', 'sex', 'race']


## Step 5 — Fairness Primary Metric Compute Equal Opportunity Difference 


Now we answer the central fairness question: **does the model correctly identify qualified people at the same rate across groups?**

"Qualified" here means "actually earns >50K" — the positive class in our target. Among the people who actually earn >$50K, the model should find them at roughly the same rate regardless of sex.

The metric that captures this is **True Positive Rate (TPR)** per group — sometimes called sensitivity or recall

- TPR(Male) = fraction of men earning 50K whom the model correctly predicts as > 50K
- TPR(Female) = fraction of women earning > 50K whom the model correctly predicts as >50K

**Equal Opportunity Difference (EOD)** is just the gap between them:

In this dataset, Male is the *privileged* group (higher base rate, better historical outcomes), so we compute `EOD = TPR(Female) − TPR(Male)`. A negative EOD means the model is worse at identifying qualified women than qualified men.

### Why EOD over other metrics

In Step 2 we saw the base rate gap is huge: 31% of men earn >$50K vs 11% of women. Many fairness metrics (like Demographic Parity) punish the model for this gap, but the gap isn't the model's fault — it's in the data. EOD looks only at *qualified* people within each group, which strips out the base-rate effect and isolates the question we actually care about: once someone is qualified, does the model recognize it equally well?

### Thresholds (from the README)

| Value | Level | Meaning |
|---|---|---|
| \|EOD\| ≤ 0.10 | PASS | Groups have similar TPR |
| 0.10 < \|EOD\| ≤ 0.15 | WARNING | Moderate gap — worth investigating |
| \|EOD\| > 0.15 | FLAG | Large gap — needs attention before deployment |

In [47]:
from fairlearn.metrics import MetricFrame, true_positive_rate

# ─── Compute TPR per group using fairlearn's MetricFrame ─────────────
# MetricFrame is the workhorse for per-group fairness evaluation.
# It takes: a metric function, y_true, y_pred, and a sensitive feature,
# and computes the metric separately for each group.
tpr_frame = MetricFrame(
    metrics=true_positive_rate,
    y_true=results["income_actual"],
    y_pred=results["income_predicted"],
    sensitive_features=results["sex"],
)

# Per-group TPR values
tpr_by_group = tpr_frame.by_group
print("True Positive Rate by sex:")
print(tpr_by_group.round(3))

# ─── Compute Equal Opportunity Difference ────────────────────────────
# EOD = TPR(Female) − TPR(Male)
# Male is the "privileged" group here (higher base rate in this dataset)
tpr_male = tpr_by_group["Male"]
tpr_female = tpr_by_group["Female"]
eod = tpr_female - tpr_male

print(f"\nTPR(Male):   {tpr_male:.3f}")
print(f"TPR(Female): {tpr_female:.3f}")
print(f"\nEqual Opportunity Difference (Female − Male): {eod:+.3f}")

# ─── Apply thresholds from the README ────────────────────────────────
abs_eod = abs(eod)
if abs_eod <= 0.10:
    status = "PASS"
    interpretation = "Groups have similar True Positive Rate."
elif abs_eod <= 0.15:
    status = "WARNING"
    interpretation = "Moderate TPR gap — worth investigating."
else:
    status = "FLAG"
    interpretation = "Large TPR gap — needs attention before deployment."

print(f"\nStatus: {status}")
print(f"→ {interpretation}")

# Also name which group is disadvantaged (the sign of EOD matters!)
if eod < 0:
    print(f"→ Female group is UNDER-identified: the model catches qualified women at a {abs_eod:.1%} lower rate than qualified men.")
elif eod > 0:
    print(f"→ Male group is UNDER-identified: the model catches qualified men at a {abs_eod:.1%} lower rate than qualified women.")

True Positive Rate by sex:
sex
Female    0.473
Male      0.606
Name: true_positive_rate, dtype: float64

TPR(Male):   0.606
TPR(Female): 0.473

Equal Opportunity Difference (Female − Male): -0.133

Status: WARNING
→ Moderate TPR gap — worth investigating.
→ Female group is UNDER-identified: the model catches qualified women at a 13.3% lower rate than qualified men.


In [48]:
# Store the EOD result in a format matching the JSON schema
eod_result = {
    "value":  round(float(eod), 4),
    "status": status,
}
print("EOD result dict (to go into the final JSON report):")
print(eod_result)

EOD result dict (to go into the final JSON report):
{'value': -0.1331, 'status': 'WARNING'}
